# Hypothesis testing: urgency associations

Experimental output. Not a clinical decision.

Run from a kernel whose working directory is the repository root with the pinned dependencies installed. All analytical logic is imported from `src/hypothesis_testing.py`. This notebook states the nine case-study hypotheses, examines their assumptions, and reports omnibus tests, effect sizes, corrected post-hoc comparisons, and a patient-level sensitivity analysis. Individual red flags and comorbidities are explicitly secondary tests within the corresponding main family.

This is observational, exploratory work following full-cohort EDA. Associations with clinician-assigned urgency are not causal effects or evidence of predictive or clinical validity. No train/test split or model fitting occurs here.

## Analysis plan and assumptions

- **Significance:** alpha is 0.05. The result `p_value` is the method's nominal p-value. Omnibus `p_value_holm` and `decision` use Holm across all primary and secondary omnibus tests within each cohort. `decision_raw` is retained to separate nominal from adjusted detection. Post-hoc tests are gated by adjusted omnibus detection, and Holm applies to all contrasts within that parent/cohort, including all category pairs and urgency outcomes for categorical comparisons. This is not global error control across all selected post-hoc families. [Holm implementation](https://www.statsmodels.org/v0.14.4/generated/statsmodels.stats.multitest.multipletests.html).
- **Continuous exposures:** Shapiro–Wilk is checked separately within each urgency class, using a seeded random subset capped at 5,000 when needed because larger-sample Shapiro p-values can be inaccurate. Shapiro H0 is within-group normality; median-centred Levene H0 is equal group variances. Non-rejection does not prove an assumption. ANOVA is used only when both diagnostics are not rejected and the no-repeated-patient check passes. Otherwise use Kruskal–Wallis, with at least five observations in each urgency class. With unequal shapes, interpret it as a rank/distribution comparison rather than a median-only test. [Shapiro](https://docs.scipy.org/doc/scipy-1.15.2/reference/generated/scipy.stats.shapiro.html), [Kruskal–Wallis](https://docs.scipy.org/doc/scipy-1.15.2/reference/generated/scipy.stats.kruskal.html).
- **Categorical exposures:** Pearson chi-square tests independence when every expected cell is at least five. If that condition fails, retain the Pearson statistic but calibrate its null distribution by fixed-margin conditional Monte Carlo with the configured seed. The CSV records the expected-count minimum, violations, resample count, and Monte Carlo standard error. Normality and variance checks are not applicable to these tables. [SciPy contingency tests](https://docs.scipy.org/doc/scipy-1.15.2/reference/generated/scipy.stats.chi2_contingency.html).
- **Post-hoc:** Kruskal–Wallis uses two-sided Dunn comparisons with pooled ranks and tie correction. ANOVA uses Tukey HSD; Tukey p-values already control family-wise error, so the additionally requested Holm is conservative. Categorical post-hoc contrasts compare the proportion of each urgency class between every exposure-category pair. Sparse expected or observed counts trigger two-sided Fisher exact instead of an invalid two-proportion z approximation. [Dunn](https://scikit-posthocs.readthedocs.io/en/latest/generated/scikit_posthocs.posthoc_dunn.html), [Tukey HSD](https://www.statsmodels.org/v0.14.4/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html).
- **Effects:** Cramer's V is sqrt(chi-square / [N times min(r−1,c−1)]); ANOVA eta-squared is between-group / total sum of squares; rank epsilon-squared is H/(N−1), with finite-sample upward bias. The epsilon definition is recorded explicitly to distinguish it from adjusted rank eta-squared. Dunn comparisons additionally report observed median differences; Tukey reports mean differences; categorical contrasts report absolute urgency-proportion differences. No clinical/practical importance threshold was prespecified. [Rank effect sizes](https://easystats.github.io/effectsize/reference/rank_epsilon_squared.html).
- **Missingness:** Exclude missing exposure/target pairs per test, not jointly across vitals. Multi-value unknowns are excluded from any/item presence–absence comparisons. Explicit none remains known absence. Each CSV row includes cohort size, n used, exclusions, patients, and per-urgency counts. Symptom duration is tested in days under the inferred-seconds convention; monotonic conversion or log1p would preserve Kruskal–Wallis/Dunn ranks.
- **Repeated patients:** Full-request p-values are nominal and unadjusted for clustering; neither a rank test nor a Monte Carlo contingency test repairs this. The sensitivity analysis selects one random request per patient, with a fixed seed and stable ID ordering, **before** any variable-specific exclusions. It never substitutes another request when the selected vital is missing. A no-repeated-ID check does not establish all other independence assumptions. Compare conclusions and effect sizes without attributing changes solely to dependence: power and selection also change. This one draw is not a cluster-robust analysis.
- **Numerical limits:** A p-value printed as zero is floating-point underflow, flagged in the CSV, not a literal zero probability. Monte Carlo p-values have finite resolution. Complete grouped-outcome separation observed for a red flag remains a source-label-provenance question.


In [1]:
from src.hypothesis_testing import (
    run_hypothesis_testing,
    show_case_results,
    show_sensitivity_summary,
)

results, posthoc = run_hypothesis_testing()

Wrote 52 omnibus results and 1040 posthoc comparisons
All requests: 14,809; one request per patient: 9,792
                         hypothesis_id     decision_all decision_patient  same_conclusion
                                H1_age        reject_H0        reject_H0             True
                                H2_sex do_not_reject_H0 do_not_reject_H0             True
                          H3_complaint        reject_H0        reject_H0             True
                           H4_severity        reject_H0        reject_H0             True
                      H5_red_flags_any        reject_H0        reject_H0             True
                  H6_comorbidities_any        reject_H0        reject_H0             True
                               H7_spo2        reject_H0        reject_H0             True
                         H8_heart_rate        reject_H0        reject_H0             True
                           H9_duration        reject_H0        reject_H0           

## Age at request

Compare age distributions across the ordered urgency classes. H0/H1 below reflect the selected test: rank distributions for Kruskal–Wallis, means for ANOVA.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [2]:
show_case_results("age")

**Age at request (primary)**

H0: Age at request rank distributions are the same across urgency classes.

H1: At least one urgency class differs in Age at request rank distributions.

,analysis,exposure,n_used,n_excluded,n_patients,independence_pass,normality_pass,variance_homogeneity_pass,levene_p_value,expected_counts_pass,minimum_expected_count,test,statistic,p_value,effect_size_name,effect_size,p_value_holm,decision
0,all_requests,Age at request,14809,0,9792,False,False,False,2.135345e-30,NaN,NaN,Kruskal-Wallis,91.654079,9.666045e-20,rank_epsilon_squared,0.006189,9.666045e-19,reject_H0
1,one_request_per_patient,Age at request,9792,0,9792,True,False,False,2.227045e-18,NaN,NaN,Kruskal-Wallis,48.068400,2.059292e-10,rank_epsilon_squared,0.004909,2.265221e-09,reject_H0


**all_requests — Age at request: assumption diagnostics**

,n_total,n_checked,W,p_value,not_rejected,note
Low,5210,5000,0.91918,0.0,False,Non-rejection does not prove normality; cap av...
Moderate,4813,4813,0.922466,0.0,False,Non-rejection does not prove normality; cap av...
High,3427,3427,0.920432,0.0,False,Non-rejection does not prove normality; cap av...
Critical,1359,1359,0.930676,0.0,False,Non-rejection does not prove normality; cap av...


Use Kruskal-Wallis: normality/variance or independence requirements not met. Kruskal-Wallis does not repair patient dependence; full-cohort p is nominal. Unequal shapes permit a rank/distribution comparison, not a median-only conclusion.

**one_request_per_patient — Age at request: assumption diagnostics**

... (2 outputs left out)

**all_requests / Age at request:** Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.0062. This corresponds to 0.62% of pooled rank variation. Group medians span 33.65–42.72 years. Request-level p-values are nominal because patients repeat; consult patient sensitivity. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Age at request:** Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.0049. This corresponds to 0.49% of pooled rank variation. Group medians span 31.92–39.80 years. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 12 stored; displaying the first 12 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision
2,all_requests,H1_age,Low,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",6569,-9.074484,years,2.162261e-17,1.297357e-16,reject_H0
4,all_requests,H1_age,Moderate,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",6172,-5.461296,years,2.552830e-11,1.276415e-10,reject_H0
1,all_requests,H1_age,Low,High,NaN,"Dunn (pooled ranks, tie-corrected)",8637,-5.045289,years,2.006960e-10,8.027840e-10,reject_H0
522,one_request_per_patient,H1_age,Low,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4362,-7.881158,years,1.614193e-10,9.685157e-10,reject_H0
524,one_request_per_patient,H1_age,Moderate,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4114,-6.272671,years,1.194744e-06,5.973719e-06,reject_H0
521,one_request_per_patient,H1_age,Low,High,NaN,"Dunn (pooled ranks, tie-corrected)",5678,-2.535521,years,2.013993e-05,8.055972e-05,reject_H0
3,all_requests,H1_age,Moderate,High,NaN,"Dunn (pooled ranks, tie-corrected)",8240,-1.432101,years,1.116418e-04,3.349253e-04,reject_H0
5,all_requests,H1_age,High,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4786,-4.029196,years,2.174758e-04,4.349516e-04,reject_H0
525,one_request_per_patient,H1_age,High,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",3042,-5.345637,years,1.634697e-03,4.904091e-03,reject_H0
0,all_requests,H1_age,Low,Moderate,NaN,"Dunn (pooled ranks, tie-corrected)",10023,-3.613188,years,7.406739e-03,7.406739e-03,reject_H0


## Recorded sex

Test independence of recorded sex and urgency; use within-exposure urgency proportions for any gated post-hoc comparisons.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [3]:
show_case_results("sex")

**Recorded sex (primary)**

H0: Clinical urgency is independent of Recorded sex among known responses.

H1: Clinical urgency is associated with Recorded sex among known responses.

,analysis,exposure,n_used,n_excluded,n_patients,independence_pass,normality_pass,variance_homogeneity_pass,levene_p_value,expected_counts_pass,minimum_expected_count,test,statistic,p_value,effect_size_name,effect_size,p_value_holm,decision
0,all_requests,Recorded sex,14809,0,9792,False,NaN,NaN,NaN,True,614.573773,Chi-square of independence,0.309411,0.958249,Cramers_V,0.004571,0.958249,do_not_reject_H0
1,one_request_per_patient,Recorded sex,9792,0,9792,True,NaN,NaN,NaN,True,389.989277,Chi-square of independence,0.205660,0.976670,Cramers_V,0.004583,1.000000,do_not_reject_H0


**all_requests / Recorded sex:** Insufficient evidence after omnibus Holm correction; equality is not established. Observed Cramers_V=0.0046. Cramer's V is on an association scale from zero toward one, not a risk difference. Request-level p-values are nominal because patients repeat; consult patient sensitivity. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Recorded sex:** Insufficient evidence after omnibus Holm correction; equality is not established. Observed Cramers_V=0.0046. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 0 stored; displaying the first 0 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision


## Chief complaint

Test the full complaint-by-urgency table without pooling small complaint categories. Inspect expected counts separately in each cohort.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [4]:
show_case_results("chief_complaint")

**Chief complaint (primary)**

H0: Clinical urgency is independent of Chief complaint among known responses.

H1: Clinical urgency is associated with Chief complaint among known responses.

,analysis,exposure,n_used,n_excluded,n_patients,independence_pass,normality_pass,variance_homogeneity_pass,levene_p_value,expected_counts_pass,minimum_expected_count,test,statistic,p_value,effect_size_name,effect_size,p_value_holm,decision
0,all_requests,Chief complaint,14809,0,9792,False,NaN,NaN,NaN,True,7.616787,Chi-square of independence,1952.240422,0.00000,Cramers_V,0.209625,0.00000,reject_H0
1,one_request_per_patient,Chief complaint,9792,0,9792,True,NaN,NaN,NaN,False,4.935458,Chi-square (conditional Monte Carlo),1309.003067,0.00005,Cramers_V,0.211093,0.00035,reject_H0


**all_requests / Chief complaint:** Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.2096. Cramer's V is on an association scale from zero toward one, not a risk difference. Request-level p-values are nominal because patients repeat; consult patient sensitivity. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Chief complaint:** Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.2111. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 840 stored; displaying the first 12 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision
70,all_requests,H3_complaint,Breathing Difficulty,Cough & Cold,Low,Two-sided two-proportion z-test,2375,-0.440804,proportion,1.046601e-97,4.395722e-95,reject_H0
106,all_requests,H3_complaint,Breathing Difficulty,Skin Rash,Low,Two-sided two-proportion z-test,1596,-0.477104,proportion,4.992485e-93,2.091851e-90,reject_H0
73,all_requests,H3_complaint,Breathing Difficulty,Cough & Cold,Critical,Two-sided two-proportion z-test,2375,0.276746,proportion,1.499374e-82,6.267383e-80,reject_H0
94,all_requests,H3_complaint,Breathing Difficulty,Muscle & Joint Pain,Low,Two-sided two-proportion z-test,1804,-0.406965,proportion,2.412575e-77,1.006044e-74,reject_H0
118,all_requests,H3_complaint,Chest Pain,Cough & Cold,Low,Two-sided two-proportion z-test,2344,-0.389230,proportion,3.716938e-73,1.546246e-70,reject_H0
154,all_requests,H3_complaint,Chest Pain,Skin Rash,Low,Two-sided two-proportion z-test,1565,-0.425531,proportion,8.074997e-70,3.351124e-67,reject_H0
74,all_requests,H3_complaint,Breathing Difficulty,ENT & Eye,Low,Two-sided two-proportion z-test,1440,-0.411538,proportion,1.388020e-69,5.746402e-67,reject_H0
590,one_request_per_patient,H3_complaint,Breathing Difficulty,Cough & Cold,Low,Two-sided two-proportion z-test,1587,-0.435676,proportion,1.533650e-61,6.441331e-59,reject_H0
626,one_request_per_patient,H3_complaint,Breathing Difficulty,Skin Rash,Low,Two-sided two-proportion z-test,1056,-0.471844,proportion,3.430701e-59,1.437464e-56,reject_H0
72,all_requests,H3_complaint,Breathing Difficulty,Cough & Cold,High,Two-sided two-proportion z-test,2375,0.296653,proportion,1.011275e-58,4.176567e-56,reject_H0


## Symptom severity

Test independence of severity and urgency. Severity retains its original ordered labels, but this omnibus chi-square comparison makes no interval-scale assumption.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [5]:
show_case_results("symptom_severity")

**Symptom severity (primary)**

H0: Clinical urgency is independent of Symptom severity among known responses.

H1: Clinical urgency is associated with Symptom severity among known responses.

,analysis,exposure,n_used,n_excluded,n_patients,independence_pass,normality_pass,variance_homogeneity_pass,levene_p_value,expected_counts_pass,minimum_expected_count,test,statistic,p_value,effect_size_name,effect_size,p_value_holm,decision
0,all_requests,Symptom severity,14809,0,9792,False,NaN,NaN,NaN,True,314.031873,Chi-square of independence,9229.551951,0.0,Cramers_V,0.558229,0.0,reject_H0
1,one_request_per_patient,Symptom severity,9792,0,9792,True,NaN,NaN,NaN,True,194.509906,Chi-square of independence,6026.362424,0.0,Cramers_V,0.554724,0.0,reject_H0


**all_requests / Symptom severity:** Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.5582. Cramer's V is on an association scale from zero toward one, not a risk difference. Request-level p-values are nominal because patients repeat; consult patient sensitivity. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Symptom severity:** Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.5547. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 24 stored; displaying the first 12 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision
426,all_requests,H4_severity,Mild,Moderate,Low,Two-sided two-proportion z-test,11387,0.454249,proportion,0.000000e+00,0.000000e+00,reject_H0
430,all_requests,H4_severity,Mild,Severe,Low,Two-sided two-proportion z-test,9814,0.639697,proportion,0.000000e+00,0.000000e+00,reject_H0
432,all_requests,H4_severity,Mild,Severe,High,Two-sided two-proportion z-test,9814,-0.424977,proportion,0.000000e+00,0.000000e+00,reject_H0
433,all_requests,H4_severity,Mild,Severe,Critical,Two-sided two-proportion z-test,9814,-0.358928,proportion,0.000000e+00,0.000000e+00,reject_H0
437,all_requests,H4_severity,Moderate,Severe,Critical,Two-sided two-proportion z-test,8417,-0.335999,proportion,0.000000e+00,0.000000e+00,reject_H0
946,one_request_per_patient,H4_severity,Mild,Moderate,Low,Two-sided two-proportion z-test,7585,0.449610,proportion,0.000000e+00,0.000000e+00,reject_H0
950,one_request_per_patient,H4_severity,Mild,Severe,Low,Two-sided two-proportion z-test,6489,0.641886,proportion,0.000000e+00,0.000000e+00,reject_H0
952,one_request_per_patient,H4_severity,Mild,Severe,High,Two-sided two-proportion z-test,6489,-0.413381,proportion,0.000000e+00,0.000000e+00,reject_H0
953,one_request_per_patient,H4_severity,Mild,Severe,Critical,Fisher exact (sparse proportion comparison),6489,-0.356579,proportion,0.000000e+00,0.000000e+00,reject_H0
957,one_request_per_patient,H4_severity,Moderate,Severe,Critical,Two-sided two-proportion z-test,5510,-0.334945,proportion,8.319861e-247,6.655889e-246,reject_H0


## Red flags: any and each individual item

The primary comparison is any reported flag versus known absence. Individual flags are secondary hypotheses. A known response reporting other flags counts as absence for the focal item; missing responses never do. Inspect zero/sparse cells and the provenance of complete separation.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [6]:
show_case_results("red_flags")

**Any reported red flag (primary)**

H0: Clinical urgency is independent of Any reported red flag among known responses.

H1: Clinical urgency is associated with Any reported red flag among known responses.

**Breathing difficulty (secondary)**

H0: Clinical urgency is independent of Breathing difficulty among known responses.

H1: Clinical urgency is associated with Breathing difficulty among known responses.

**Chest pain (secondary)**

H0: Clinical urgency is independent of Chest pain among known responses.

H1: Clinical urgency is associated with Chest pain among known responses.

**Confusion (secondary)**

H0: Clinical urgency is independent of Confusion among known responses.

H1: Clinical urgency is associated with Confusion among known responses.

**Loss of consciousness (secondary)**

H0: Clinical urgency is independent of Loss of consciousness among known responses.

H1: Clinical urgency is associated with Loss of consciousness among known responses.

**Seizure (secondary)**

H0: Clinical urgency is independent of Seizure among known responses.

H1: Clinical urgency is associated with Seizure among known responses.

... (23 outputs left out)

**one_request_per_patient / Stroke signs:** Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.1377. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Thoughts of self-harm:** Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.2410. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 80 stored; displaying the first 12 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision
438,all_requests,H5_red_flags_any,Absent,Present,Low,Two-sided two-proportion z-test,14547,0.376496,proportion,0.000000e+00,0.000000e+00,reject_H0
440,all_requests,H5_red_flags_any,Absent,Present,High,Two-sided two-proportion z-test,14547,-0.324396,proportion,0.000000e+00,0.000000e+00,reject_H0
441,all_requests,H5_red_flags_any,Absent,Present,Critical,Two-sided two-proportion z-test,14547,-0.259873,proportion,0.000000e+00,0.000000e+00,reject_H0
961,one_request_per_patient,H5_red_flags_any,Absent,Present,Critical,Two-sided two-proportion z-test,9617,-0.251529,proportion,4.445538e-303,1.778215e-302,reject_H0
467,all_requests,red_flags__breathing_difficulty,Absent,Present,Critical,Two-sided two-proportion z-test,14547,-0.328097,proportion,3.820313e-296,1.528125e-295,reject_H0
960,one_request_per_patient,H5_red_flags_any,Absent,Present,High,Two-sided two-proportion z-test,9617,-0.331212,proportion,2.179412e-243,6.538236e-243,reject_H0
958,one_request_per_patient,H5_red_flags_any,Absent,Present,Low,Two-sided two-proportion z-test,9617,0.376610,proportion,1.311829e-237,2.623657e-237,reject_H0
987,one_request_per_patient,red_flags__breathing_difficulty,Absent,Present,Critical,Two-sided two-proportion z-test,9617,-0.326054,proportion,7.071512e-196,2.828605e-195,reject_H0
475,all_requests,red_flags__confusion,Absent,Present,Critical,Two-sided two-proportion z-test,14547,-0.322961,proportion,7.455321e-164,2.982128e-163,reject_H0
464,all_requests,red_flags__breathing_difficulty,Absent,Present,Low,Two-sided two-proportion z-test,14547,0.358821,proportion,3.785280e-130,1.135584e-129,reject_H0


## Comorbidities: any and each individual condition

The primary comparison is any reported condition versus known absence. Each condition is a secondary comparison among known responses. Conditions overlap; correction and interpretation account for the defined test families without assuming mutually exclusive items.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [7]:
show_case_results("comorbidities")

**Any reported comorbidity (primary)**

H0: Clinical urgency is independent of Any reported comorbidity among known responses.

H1: Clinical urgency is associated with Any reported comorbidity among known responses.

**Asthma (secondary)**

H0: Clinical urgency is independent of Asthma among known responses.

H1: Clinical urgency is associated with Asthma among known responses.

**Chronic kidney disease (secondary)**

H0: Clinical urgency is independent of Chronic kidney disease among known responses.

H1: Clinical urgency is associated with Chronic kidney disease among known responses.

**Diabetes (secondary)**

H0: Clinical urgency is independent of Diabetes among known responses.

H1: Clinical urgency is associated with Diabetes among known responses.

**HIV (secondary)**

H0: Clinical urgency is independent of HIV among known responses.

H1: Clinical urgency is associated with HIV among known responses.

**Heart disease (secondary)**

H0: Clinical urgency is independent of Heart disease among known responses.

H1: Clinical urgency is associated with Heart disease among known responses.

... (20 outputs left out)

**one_request_per_patient / Other chronic condition:** Insufficient evidence after omnibus Holm correction; equality is not established. Observed Cramers_V=0.0063. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Sickle cell disease:** Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.0527. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 48 stored; displaying the first 12 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision
442,all_requests,H6_comorbidities_any,Absent,Present,Low,Two-sided two-proportion z-test,14516,0.068729,proportion,8.345782e-18,3.338313e-17,reject_H0
445,all_requests,H6_comorbidities_any,Absent,Present,Critical,Two-sided two-proportion z-test,14516,-0.037613,proportion,7.801712e-15,2.340514e-14,reject_H0
444,all_requests,H6_comorbidities_any,Absent,Present,High,Two-sided two-proportion z-test,14516,-0.045527,proportion,1.127430e-10,2.254860e-10,reject_H0
503,all_requests,comorbidities__asthma,Absent,Present,Critical,Two-sided two-proportion z-test,14516,-0.056197,proportion,7.470788e-11,2.988315e-10,reject_H0
965,one_request_per_patient,H6_comorbidities_any,Absent,Present,Critical,Two-sided two-proportion z-test,9596,-0.039987,proportion,1.248065e-10,4.992260e-10,reject_H0
512,all_requests,comorbidities__hypertension,Absent,Present,Low,Two-sided two-proportion z-test,14516,0.059853,proportion,7.899518e-10,3.159807e-09,reject_H0
962,one_request_per_patient,H6_comorbidities_any,Absent,Present,Low,Two-sided two-proportion z-test,9596,0.061292,proportion,5.154553e-09,1.546366e-08,reject_H0
515,all_requests,comorbidities__hypertension,Absent,Present,Critical,Two-sided two-proportion z-test,14516,-0.033378,proportion,1.483847e-08,4.451540e-08,reject_H0
1023,one_request_per_patient,comorbidities__asthma,Absent,Present,Critical,Two-sided two-proportion z-test,9596,-0.071081,proportion,1.600483e-08,6.401932e-08,reject_H0
514,all_requests,comorbidities__hypertension,Absent,Present,High,Two-sided two-proportion z-test,14516,-0.045574,proportion,1.147477e-07,2.294954e-07,reject_H0


## Oxygen saturation

Use only requests with observed oxygen saturation for this test; do not drop a request because another vital is missing. Sample sizes are reported for each cohort.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [8]:
show_case_results("oxygen_saturation")

**Oxygen saturation (primary)**

H0: Oxygen saturation rank distributions are the same across urgency classes.

H1: At least one urgency class differs in Oxygen saturation rank distributions.

,analysis,exposure,n_used,n_excluded,n_patients,independence_pass,normality_pass,variance_homogeneity_pass,levene_p_value,expected_counts_pass,minimum_expected_count,test,statistic,p_value,effect_size_name,effect_size,p_value_holm,decision
0,all_requests,Oxygen saturation,13872,937,9222,False,False,False,1.175936e-283,NaN,NaN,Kruskal-Wallis,4741.644284,0.0,rank_epsilon_squared,0.341839,0.0,reject_H0
1,one_request_per_patient,Oxygen saturation,9181,611,9181,True,False,False,2.438137e-189,NaN,NaN,Kruskal-Wallis,3080.934188,0.0,rank_epsilon_squared,0.335614,0.0,reject_H0


**all_requests — Oxygen saturation: assumption diagnostics**

,n_total,n_checked,W,p_value,not_rejected,note
Low,4893,4893,0.937875,0.0,False,Non-rejection does not prove normality; cap av...
Moderate,4517,4517,0.957544,0.0,False,Non-rejection does not prove normality; cap av...
High,3202,3202,0.940604,0.0,False,Non-rejection does not prove normality; cap av...
Critical,1260,1260,0.860525,0.0,False,Non-rejection does not prove normality; cap av...


Use Kruskal-Wallis: normality/variance or independence requirements not met. Kruskal-Wallis does not repair patient dependence; full-cohort p is nominal. Unequal shapes permit a rank/distribution comparison, not a median-only conclusion.

**one_request_per_patient — Oxygen saturation: assumption diagnostics**

... (2 outputs left out)

**all_requests / Oxygen saturation:** Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.3418. This corresponds to 34.18% of pooled rank variation. Group medians span 94.00–98.00 percentage points. Request-level p-values are nominal because patients repeat; consult patient sensitivity. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Oxygen saturation:** Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.3356. This corresponds to 33.56% of pooled rank variation. Group medians span 94.00–98.00 percentage points. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 12 stored; displaying the first 12 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision
447,all_requests,H7_spo2,Low,High,NaN,"Dunn (pooled ranks, tie-corrected)",8095,2.0,percentage points,0.000000e+00,0.000000e+00,reject_H0
448,all_requests,H7_spo2,Low,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",6153,4.0,percentage points,0.000000e+00,0.000000e+00,reject_H0
450,all_requests,H7_spo2,Moderate,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",5777,3.0,percentage points,0.000000e+00,0.000000e+00,reject_H0
967,one_request_per_patient,H7_spo2,Low,High,NaN,"Dunn (pooled ranks, tie-corrected)",5324,2.0,percentage points,0.000000e+00,0.000000e+00,reject_H0
968,one_request_per_patient,H7_spo2,Low,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4081,4.0,percentage points,0.000000e+00,0.000000e+00,reject_H0
970,one_request_per_patient,H7_spo2,Moderate,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",3857,3.0,percentage points,1.102336e-222,4.409345e-222,reject_H0
449,all_requests,H7_spo2,Moderate,High,NaN,"Dunn (pooled ranks, tie-corrected)",7719,1.0,percentage points,1.737850e-197,5.213551e-197,reject_H0
446,all_requests,H7_spo2,Low,Moderate,NaN,"Dunn (pooled ranks, tie-corrected)",9410,1.0,percentage points,9.704166e-142,1.940833e-141,reject_H0
969,one_request_per_patient,H7_spo2,Moderate,High,NaN,"Dunn (pooled ranks, tie-corrected)",5100,1.0,percentage points,2.108313e-126,6.324938e-126,reject_H0
966,one_request_per_patient,H7_spo2,Low,Moderate,NaN,"Dunn (pooled ranks, tie-corrected)",6344,1.0,percentage points,2.207319e-96,4.414638e-96,reject_H0


## Heart rate

Use only requests with observed heart rate for this test. Compare group diagnostics and rank effect sizes as well as corrected p-values.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [9]:
show_case_results("heart_rate")

**Heart rate (primary)**

H0: Heart rate rank distributions are the same across urgency classes.

H1: At least one urgency class differs in Heart rate rank distributions.

,analysis,exposure,n_used,n_excluded,n_patients,independence_pass,normality_pass,variance_homogeneity_pass,levene_p_value,expected_counts_pass,minimum_expected_count,test,statistic,p_value,effect_size_name,effect_size,p_value_holm,decision
0,all_requests,Heart rate,14434,375,9588,False,False,False,3.507758e-57,NaN,NaN,Kruskal-Wallis,5270.757252,0.0,rank_epsilon_squared,0.365188,0.0,reject_H0
1,one_request_per_patient,Heart rate,9541,251,9541,True,False,False,1.021458e-42,NaN,NaN,Kruskal-Wallis,3508.764392,0.0,rank_epsilon_squared,0.367795,0.0,reject_H0


**all_requests — Heart rate: assumption diagnostics**

,n_total,n_checked,W,p_value,not_rejected,note
Low,5090,5000,0.992276,0.0,False,Non-rejection does not prove normality; cap av...
Moderate,4715,4715,0.988141,0.0,False,Non-rejection does not prove normality; cap av...
High,3328,3328,0.991853,0.0,False,Non-rejection does not prove normality; cap av...
Critical,1301,1301,0.989509,0.0,False,Non-rejection does not prove normality; cap av...


Use Kruskal-Wallis: normality/variance or independence requirements not met. Kruskal-Wallis does not repair patient dependence; full-cohort p is nominal. Unequal shapes permit a rank/distribution comparison, not a median-only conclusion.

**one_request_per_patient — Heart rate: assumption diagnostics**

... (2 outputs left out)

**all_requests / Heart rate:** Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.3652. This corresponds to 36.52% of pooled rank variation. Group medians span 78.00–106.00 beats/min. Request-level p-values are nominal because patients repeat; consult patient sensitivity. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Heart rate:** Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.3678. This corresponds to 36.78% of pooled rank variation. Group medians span 78.00–107.00 beats/min. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 12 stored; displaying the first 12 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision
453,all_requests,H8_heart_rate,Low,High,NaN,"Dunn (pooled ranks, tie-corrected)",8418,-17.0,beats/min,0.000000e+00,0.000000e+00,reject_H0
454,all_requests,H8_heart_rate,Low,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",6391,-28.0,beats/min,0.000000e+00,0.000000e+00,reject_H0
456,all_requests,H8_heart_rate,Moderate,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",6016,-21.0,beats/min,0.000000e+00,0.000000e+00,reject_H0
973,one_request_per_patient,H8_heart_rate,Low,High,NaN,"Dunn (pooled ranks, tie-corrected)",5537,-17.0,beats/min,0.000000e+00,0.000000e+00,reject_H0
974,one_request_per_patient,H8_heart_rate,Low,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4244,-29.0,beats/min,0.000000e+00,0.000000e+00,reject_H0
976,one_request_per_patient,H8_heart_rate,Moderate,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4004,-22.0,beats/min,1.193647e-234,4.774588e-234,reject_H0
452,all_requests,H8_heart_rate,Low,Moderate,NaN,"Dunn (pooled ranks, tie-corrected)",9805,-7.0,beats/min,8.517193e-197,2.555158e-196,reject_H0
455,all_requests,H8_heart_rate,Moderate,High,NaN,"Dunn (pooled ranks, tie-corrected)",8043,-10.0,beats/min,1.652104e-173,3.304208e-173,reject_H0
972,one_request_per_patient,H8_heart_rate,Low,Moderate,NaN,"Dunn (pooled ranks, tie-corrected)",6608,-7.0,beats/min,8.918464e-135,2.675539e-134,reject_H0
975,one_request_per_patient,H8_heart_rate,Moderate,High,NaN,"Dunn (pooled ranks, tie-corrected)",5297,-10.0,beats/min,6.341927e-121,1.268385e-120,reject_H0


## Symptom duration

Analyse the cleaned duration in days without trimming its right tail. Units remain inferred from source values. Kruskal–Wallis is invariant to positive unit conversion and log1p, but its omnibus rejection does not identify the differing pairs.

For each exposure below, hypotheses and diagnostic summaries are followed by n used, test statistic, nominal/corrected p-values, effect size, decision, and plain-language interpretation. Full group diagnostics and every post-hoc contrast are preserved in the CSV artifacts.


In [10]:
show_case_results("symptom_duration")

**Symptom duration (primary)**

H0: Symptom duration rank distributions are the same across urgency classes.

H1: At least one urgency class differs in Symptom duration rank distributions.

,analysis,exposure,n_used,n_excluded,n_patients,independence_pass,normality_pass,variance_homogeneity_pass,levene_p_value,expected_counts_pass,minimum_expected_count,test,statistic,p_value,effect_size_name,effect_size,p_value_holm,decision
0,all_requests,Symptom duration,14809,0,9792,False,False,False,3.824161e-08,NaN,NaN,Kruskal-Wallis,760.766888,1.395420e-164,rank_epsilon_squared,0.051375,2.372214e-163,reject_H0
1,one_request_per_patient,Symptom duration,9792,0,9792,True,False,False,1.456547e-07,NaN,NaN,Kruskal-Wallis,547.231267,2.767306e-118,rank_epsilon_squared,0.055891,4.981151e-117,reject_H0


**all_requests — Symptom duration: assumption diagnostics**

,n_total,n_checked,W,p_value,not_rejected,note
Low,5210,5000,0.535059,0.0,False,Non-rejection does not prove normality; cap av...
Moderate,4813,4813,0.543995,0.0,False,Non-rejection does not prove normality; cap av...
High,3427,3427,0.480109,0.0,False,Non-rejection does not prove normality; cap av...
Critical,1359,1359,0.389133,0.0,False,Non-rejection does not prove normality; cap av...


Use Kruskal-Wallis: normality/variance or independence requirements not met. Kruskal-Wallis does not repair patient dependence; full-cohort p is nominal. Unequal shapes permit a rank/distribution comparison, not a median-only conclusion.

**one_request_per_patient — Symptom duration: assumption diagnostics**

... (2 outputs left out)

**all_requests / Symptom duration:** Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.0514. This corresponds to 5.14% of pooled rank variation. Group medians span 0.77–3.12 days (seconds inferred). Request-level p-values are nominal because patients repeat; consult patient sensitivity. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**one_request_per_patient / Symptom duration:** Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.0559. This corresponds to 5.59% of pooled rank variation. Group medians span 0.70–3.10 days (seconds inferred). One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

**Posthoc comparisons:** 12 stored; displaying the first 12 by corrected p-value below. All contrasts, denominators, effects, and decisions are in `artifacts/posthoc_results.csv`.

,analysis,hypothesis_id,group_a,group_b,urgency_category,test,n_used,effect_size,effect_units,p_value,p_value_holm,decision
460,all_requests,H9_duration,Low,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",6569,2.345833,days (seconds inferred),3.504518e-133,2.102711e-132,reject_H0
462,all_requests,H9_duration,Moderate,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",6172,1.995833,days (seconds inferred),2.652935e-95,1.326468e-94,reject_H0
980,one_request_per_patient,H9_duration,Low,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4362,2.399306,days (seconds inferred),1.882342e-94,1.129405e-93,reject_H0
982,one_request_per_patient,H9_duration,Moderate,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4114,1.982639,days (seconds inferred),8.956689e-67,4.478344e-66,reject_H0
459,all_requests,H9_duration,Low,High,NaN,"Dunn (pooled ranks, tie-corrected)",8637,1.189583,days (seconds inferred),6.217876e-66,2.487150e-65,reject_H0
979,one_request_per_patient,H9_duration,Low,High,NaN,"Dunn (pooled ranks, tie-corrected)",5678,1.289583,days (seconds inferred),2.019973e-50,8.079892e-50,reject_H0
461,all_requests,H9_duration,Moderate,High,NaN,"Dunn (pooled ranks, tie-corrected)",8240,0.839583,days (seconds inferred),1.583266e-32,4.749798e-32,reject_H0
463,all_requests,H9_duration,High,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",4786,1.156250,days (seconds inferred),5.999926e-31,1.199985e-30,reject_H0
981,one_request_per_patient,H9_duration,Moderate,High,NaN,"Dunn (pooled ranks, tie-corrected)",5430,0.872917,days (seconds inferred),7.760389e-25,2.328117e-24,reject_H0
983,one_request_per_patient,H9_duration,High,Critical,NaN,"Dunn (pooled ranks, tie-corrected)",3042,1.109722,days (seconds inferred),8.612420e-21,1.722484e-20,reject_H0


## Sensitivity comparison and plain-language interpretation

The final section is generated from the saved artifacts. Compare adjusted decisions and observed effect magnitudes, while recognising that the patient sample has fewer observations and can change the tested population. Agreement in this one sample is not proof of stability across alternative patient selections. Effects of different types are not subtracted. Statistical detection does not establish practical importance or causation.


In [11]:
show_sensitivity_summary()

,hypothesis_id,family,tier,n_used_all,test_all,p_value_all,p_value_holm_all,effect_size_name_all,effect_size_all,decision_raw_all,...,p_value_holm_patient,effect_size_name_patient,effect_size_patient,decision_raw_patient,decision_patient,same_conclusion,same_raw_conclusion,effect_sizes_comparable,effect_difference_patient_minus_all,interpretation
0,H1_age,age,primary,14809,Kruskal-Wallis,9.666045e-20,9.666045e-19,rank_epsilon_squared,0.006189,reject_H0,...,2.265221e-09,rank_epsilon_squared,0.004909,reject_H0,reject_H0,True,True,True,-0.001280,Adjusted decisions agree in this one seeded pa...
1,H2_sex,sex,primary,14809,Chi-square of independence,9.582489e-01,9.582489e-01,Cramers_V,0.004571,do_not_reject_H0,...,1.000000e+00,Cramers_V,0.004583,do_not_reject_H0,do_not_reject_H0,True,True,True,0.000012,Adjusted decisions agree in this one seeded pa...
2,H3_complaint,chief_complaint,primary,14809,Chi-square of independence,0.000000e+00,0.000000e+00,Cramers_V,0.209625,reject_H0,...,3.500000e-04,Cramers_V,0.211093,reject_H0,reject_H0,True,True,True,0.001468,Adjusted decisions agree in this one seeded pa...
3,H4_severity,symptom_severity,primary,14809,Chi-square of independence,0.000000e+00,0.000000e+00,Cramers_V,0.558229,reject_H0,...,0.000000e+00,Cramers_V,0.554724,reject_H0,reject_H0,True,True,True,-0.003505,Adjusted decisions agree in this one seeded pa...
4,H5_red_flags_any,red_flags,primary,14547,Chi-square of independence,0.000000e+00,0.000000e+00,Cramers_V,0.564987,reject_H0,...,0.000000e+00,Cramers_V,0.562825,reject_H0,reject_H0,True,True,True,-0.002161,Adjusted decisions agree in this one seeded pa...
5,H6_comorbidities_any,comorbidities,primary,14516,Chi-square of independence,1.759774e-29,1.935752e-28,Cramers_V,0.097114,reject_H0,...,4.575784e-14,Cramers_V,0.085548,reject_H0,reject_H0,True,True,True,-0.011565,Adjusted decisions agree in this one seeded pa...
6,H7_spo2,oxygen_saturation,primary,13872,Kruskal-Wallis,0.000000e+00,0.000000e+00,rank_epsilon_squared,0.341839,reject_H0,...,0.000000e+00,rank_epsilon_squared,0.335614,reject_H0,reject_H0,True,True,True,-0.006225,Adjusted decisions agree in this one seeded pa...
7,H8_heart_rate,heart_rate,primary,14434,Kruskal-Wallis,0.000000e+00,0.000000e+00,rank_epsilon_squared,0.365188,reject_H0,...,0.000000e+00,rank_epsilon_squared,0.367795,reject_H0,reject_H0,True,True,True,0.002607,Adjusted decisions agree in this one seeded pa...
8,H9_duration,symptom_duration,primary,14809,Kruskal-Wallis,1.395420e-164,2.372214e-163,rank_epsilon_squared,0.051375,reject_H0,...,4.981151e-117,rank_epsilon_squared,0.055891,reject_H0,reject_H0,True,True,True,0.004516,Adjusted decisions agree in this one seeded pa...
9,red_flags__breathing_difficulty,red_flags,secondary,14547,Chi-square of independence,0.000000e+00,0.000000e+00,Cramers_V,0.368262,reject_H0,...,6.446156e-286,Cramers_V,0.371361,reject_H0,reject_H0,True,True,True,0.003100,Adjusted decisions agree in this one seeded pa...


# Hypothesis-testing interpretation



Alpha: 0.05. Full cohort: 14,809 requests; sensitivity: 9,792 patients, one request each.



Full-cohort p-values are nominal because repeated requests are dependent. Kruskal-Wallis does not repair that violation. The sensitivity results are the more defensible comparison for within-patient repetition, conditional on this one seeded selection; patient-level confounding and other dependence remain.



## Nine primary comparisons



- **Age at request**: Kruskal-Wallis, n=9,792; Holm p=2.265e-09; rank_epsilon_squared=0.0049; reject_H0. Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.0049. This corresponds to 0.49% of pooled rank variation. Group medians span 31.92–39.80 years. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

- **Recorded sex**: Chi-square of independence, n=9,792; Holm p=1; Cramers_V=0.0046; do_not_reject_H0. Insufficient evidence after omnibus Holm correction; equality is not established. Observed Cramers_V=0.0046. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

- **Chief complaint**: Chi-square (conditional Monte Carlo), n=9,792; Holm p=0.00035; Cramers_V=0.2111; reject_H0. Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.2111. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

- **Symptom severity**: Chi-square of independence, n=9,792; Holm p=0; Cramers_V=0.5547; reject_H0. Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.5547. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

- **Any reported red flag**: Chi-square of independence, n=9,617; Holm p=0; Cramers_V=0.5628; reject_H0. Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.5628. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

- **Any reported comorbidity**: Chi-square of independence, n=9,596; Holm p=4.576e-14; Cramers_V=0.0855; reject_H0. Evidence of an association/difference after omnibus Holm correction. Observed Cramers_V=0.0855. Cramer's V is on an association scale from zero toward one, not a risk difference. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

- **Oxygen saturation**: Kruskal-Wallis, n=9,181; Holm p=0; rank_epsilon_squared=0.3356; reject_H0. Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.3356. This corresponds to 33.56% of pooled rank variation. Group medians span 94.00–98.00 percentage points. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

- **Heart rate**: Kruskal-Wallis, n=9,541; Holm p=0; rank_epsilon_squared=0.3678; reject_H0. Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.3678. This corresponds to 36.78% of pooled rank variation. Group medians span 78.00–107.00 beats/min. One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.

- **Symptom duration**: Kruskal-Wallis, n=9,792; Holm p=4.981e-117; rank_epsilon_squared=0.0559; reject_H0. Evidence of an association/difference after omnibus Holm correction. Observed rank_epsilon_squared=0.0559. This corresponds to 5.59% of pooled rank variation. Group medians span 0.70–3.10 days (seconds inferred). One request per patient addresses within-patient repetition. Statistical significance does not establish practical/clinical importance; no practical threshold was prespecified. Association does not imply causation.



Sensitivity comparisons detected after omnibus Holm: Age at request, Chief complaint, Symptom severity, Any reported red flag, Any reported comorbidity, Oxygen saturation, Heart rate, Symptom duration, Breathing difficulty, Chest pain, Confusion, Loss of consciousness, Seizure, Severe allergic reaction, Severe bleeding, Stroke signs, Thoughts of self-harm, Asthma, Diabetes, Heart disease, Hypertension, Sickle cell disease.

Adjusted conclusions differing across cohorts: none.



## Interpretation limits



Holm controls the defined omnibus family within each analysis; posthoc Holm is local to each parent hypothesis, not a claim of global error control across every selected posthoc contrast. Tukey uses additional conservative Holm.

Statistical detection and practical importance differ. Numeric comparisons report observed median ranges, and proportion posthocs report absolute differences, but no clinically meaningful threshold was specified. Non-rejection does not prove equality. These observational analyses do not establish causation.

Missing values are excluded per variable/target pair. Unknown multivalue responses are excluded, not relabelled absent. Duration is tested in days under the preparation's inferred-seconds convention; KW ranks are unchanged by positive unit conversion or log1p transformation.

Sparse tables use conditional Monte Carlo calibration; sparse posthoc proportion contrasts use Fisher exact tests. Complete grouped-outcome separation in the self-harm flag remains a reason to audit source label construction. A floating-point p=0 is underflow, not a literal zero probability.
